# 🛒 Customer Segmentation using K-Means Clustering

**Objective:** Segment mall customers into distinct groups based on their Annual Income and Spending Score using unsupervised machine learning.

**Dataset:** Mall Customer Segmentation Data (200 customers, 5 features)

**Algorithm:** K-Means Clustering

---

## Step 1: Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import joblib
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('✅ Libraries loaded successfully!')

In [ ]:
# Load the dataset
df = pd.read_csv('../data/Mall_Customers.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nFirst 5 rows:')
df.head()

## Step 2: Data Exploration

In [ ]:
# Basic info
print('=== Dataset Info ===')
print(f'Shape: {df.shape}')
print(f'\nColumn Types:')
print(df.dtypes)
print(f'\nMissing Values:')
print(df.isnull().sum())
print(f'\nDuplicate Rows: {df.duplicated().sum()}')

In [ ]:
# Descriptive statistics
df.describe().round(2)

In [ ]:
# Gender distribution
print('Gender Distribution:')
print(df['Gender'].value_counts())
print(f'\nGender Percentage:')
print(df['Gender'].value_counts(normalize=True).round(3) * 100)

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Distribution plots for all numeric features
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

features = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
colors = ['#6C5CE7', '#00B894', '#E17055']

for i, (feat, color) in enumerate(zip(features, colors)):
    axes[i].hist(df[feat], bins=20, color=color, edgecolor='white', alpha=0.8)
    axes[i].set_title(f'Distribution of {feat}', fontweight='bold')
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Frequency')
    axes[i].axvline(df[feat].mean(), color='red', linestyle='--', label=f'Mean: {df[feat].mean():.1f}')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Gender distribution pie chart
fig = px.pie(df, names='Gender', title='👥 Gender Distribution',
             color_discrete_sequence=['#6C5CE7', '#00B894'],
             hole=0.4)
fig.update_layout(template='plotly_dark', paper_bgcolor='rgba(0,0,0,0)')
fig.show()

In [ ]:
# Box plots by Gender
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, feat in enumerate(features):
    sns.boxplot(data=df, x='Gender', y=feat, ax=axes[i],
                palette=['#6C5CE7', '#00B894'])
    axes[i].set_title(f'{feat} by Gender', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
le = LabelEncoder()
df_encoded = df.copy()
df_encoded['Gender_Encoded'] = le.fit_transform(df['Gender'])

numeric_cols = df_encoded.select_dtypes(include=[np.number])
corr = numeric_cols.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='viridis', center=0,
            square=True, linewidths=2, fmt='.3f',
            annot_kws={'size': 12})
plt.title('🔗 Feature Correlation Heatmap', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot
sns.pairplot(df, hue='Gender', palette=['#6C5CE7', '#00B894'],
             diag_kind='kde', plot_kws={'alpha': 0.7})
plt.suptitle('🔍 Feature Pair Analysis', y=1.02, fontsize=16, fontweight='bold')
plt.show()

In [ ]:
# Scatter: Income vs Spending Score (the key relationship)
fig = px.scatter(df, x='Annual Income (k$)', y='Spending Score (1-100)',
                 color='Gender', size='Age',
                 color_discrete_sequence=['#6C5CE7', '#00B894'],
                 title='💰 Income vs Spending Score',
                 template='plotly_dark', opacity=0.8)
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)')
fig.show()

print('\n💡 Key Insight: We can visually see ~5 distinct groups forming!')

## Step 4: Feature Selection & Scaling

In [ ]:
# Select features for clustering
X = df[['Annual Income (k$)', 'Spending Score (1-100)']].copy()

print('Selected Features:')
print(f'  - Annual Income (k$)')
print(f'  - Spending Score (1-100)')
print(f'\nShape: {X.shape}')
print(f'\nSample data:')
X.head()

In [ ]:
# Standardize the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('✅ Features scaled using StandardScaler')
print(f'\nScaled data sample (first 5 rows):')
print(X_scaled[:5])
print(f'\nMean after scaling: {X_scaled.mean(axis=0).round(6)}')
print(f'Std after scaling:  {X_scaled.std(axis=0).round(6)}')

## Step 5: Finding Optimal K — Elbow Method & Silhouette Analysis

In [ ]:
# Elbow Method
k_range = range(2, 11)
wcss = []
silhouette_scores = []

for k in k_range:
    kmeans = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels = kmeans.fit_predict(X_scaled)
    wcss.append(kmeans.inertia_)
    sil_score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(sil_score)
    print(f'K={k:2d} | WCSS={kmeans.inertia_:8.2f} | Silhouette={sil_score:.4f}')

In [ ]:
# Plot Elbow Method & Silhouette Scores side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Elbow plot
axes[0].plot(list(k_range), wcss, 'o-', color='#6C5CE7', linewidth=3, markersize=10)
axes[0].axvline(x=5, color='#E17055', linestyle='--', linewidth=2, label='K=5 (Elbow)')
axes[0].set_title('📐 Elbow Method', fontsize=16, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (K)', fontsize=13)
axes[0].set_ylabel('WCSS', fontsize=13)
axes[0].legend(fontsize=12)
axes[0].grid(True, alpha=0.3)

# Silhouette plot
colors_bar = ['#6C5CE7' if k == 5 else '#A29BFE' for k in k_range]
axes[1].bar(list(k_range), silhouette_scores, color=colors_bar, edgecolor='white')
axes[1].set_title('🎯 Silhouette Scores', fontsize=16, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (K)', fontsize=13)
axes[1].set_ylabel('Silhouette Score', fontsize=13)
axes[1].grid(True, alpha=0.3)

for i, (k, score) in enumerate(zip(k_range, silhouette_scores)):
    axes[1].text(k, score + 0.005, f'{score:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

best_k = list(k_range)[np.argmax(silhouette_scores)]
print(f'\n🏆 Best K by Silhouette Score: {best_k} (score = {max(silhouette_scores):.4f})')
print(f'📐 Elbow suggests K = 5')

## Step 6: Train K-Means Model (K=5)

In [ ]:
# Train the final model with K=5
OPTIMAL_K = 5

kmeans = KMeans(
    n_clusters=OPTIMAL_K,
    init='k-means++',
    n_init=10,
    max_iter=300,
    random_state=42
)

cluster_labels = kmeans.fit_predict(X_scaled)

# Add cluster labels to dataframe
df['Cluster'] = cluster_labels

print(f'✅ K-Means trained with K={OPTIMAL_K}')
print(f'\nCluster Distribution:')
print(df['Cluster'].value_counts().sort_index())
print(f'\nInertia (WCSS): {kmeans.inertia_:.2f}')
print(f'Silhouette Score: {silhouette_score(X_scaled, cluster_labels):.4f}')

In [ ]:
# Get centroids in original scale
centroids_scaled = kmeans.cluster_centers_
centroids_original = scaler.inverse_transform(centroids_scaled)

print('Cluster Centroids (Original Scale):')
centroid_df = pd.DataFrame(
    centroids_original,
    columns=['Annual Income (k$)', 'Spending Score (1-100)']
)
centroid_df.index.name = 'Cluster'
centroid_df.round(2)

In [ ]:
# 2D Cluster Visualization
cluster_colors = ['#6C5CE7', '#00B894', '#E17055', '#0984E3', '#FDCB6E']

plt.figure(figsize=(12, 8))

for i in range(OPTIMAL_K):
    cluster_data = df[df['Cluster'] == i]
    plt.scatter(
        cluster_data['Annual Income (k$)'],
        cluster_data['Spending Score (1-100)'],
        s=100, c=cluster_colors[i], label=f'Cluster {i}',
        edgecolors='white', linewidth=1, alpha=0.8
    )

# Plot centroids
plt.scatter(
    centroids_original[:, 0], centroids_original[:, 1],
    s=300, c='red', marker='X', edgecolors='black',
    linewidth=2, label='Centroids', zorder=5
)

plt.title('🔮 Customer Segments — K-Means Clustering (K=5)', fontsize=16, fontweight='bold')
plt.xlabel('Annual Income (k$)', fontsize=13)
plt.ylabel('Spending Score (1-100)', fontsize=13)
plt.legend(fontsize=11, loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Interactive 2D Plotly scatter
df['Cluster_str'] = df['Cluster'].astype(str)

fig = px.scatter(
    df, x='Annual Income (k$)', y='Spending Score (1-100)',
    color='Cluster_str',
    color_discrete_sequence=cluster_colors,
    title='🔮 Interactive Customer Segments',
    template='plotly_dark',
    hover_data=['Age', 'Gender'],
    opacity=0.85
)

fig.update_traces(marker=dict(size=12, line=dict(width=1, color='white')))

# Add centroids
fig.add_trace(go.Scatter(
    x=centroids_original[:, 0], y=centroids_original[:, 1],
    mode='markers', marker=dict(size=20, color='white', symbol='x',
                                 line=dict(width=3, color='black')),
    name='Centroids'
))

fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', height=600)
fig.show()

In [ ]:
# 3D scatter plot
fig = px.scatter_3d(
    df, x='Annual Income (k$)', y='Spending Score (1-100)', z='Age',
    color='Cluster_str',
    color_discrete_sequence=cluster_colors,
    title='🌐 3D Customer Segments',
    template='plotly_dark',
    opacity=0.85
)

fig.update_traces(marker=dict(size=6, line=dict(width=0.5, color='white')))
fig.update_layout(paper_bgcolor='rgba(0,0,0,0)', height=600)
fig.show()

## Step 7: Cluster Profiling

In [ ]:
# Cluster summary statistics
cluster_profile = df.groupby('Cluster').agg({
    'CustomerID': 'count',
    'Age': 'mean',
    'Annual Income (k$)': 'mean',
    'Spending Score (1-100)': 'mean',
    'Gender': lambda x: x.value_counts().to_dict()
}).round(1)

cluster_profile.columns = ['Count', 'Avg Age', 'Avg Income (k$)', 'Avg Spending Score', 'Gender Split']

print('📋 Cluster Profiles:')
cluster_profile

In [ ]:
# Business-friendly cluster naming
cluster_names = {
    0: '💰 High Income, High Spenders',
    1: '🎯 Moderate Income, Moderate Spenders',
    2: '📉 High Income, Low Spenders',
    3: '🛍️ Low Income, High Spenders',
    4: '💵 Low Income, Low Spenders'
}

print('\n🏷️ Business-Friendly Segment Names:')
print('=' * 50)
for cid, name in cluster_names.items():
    count = len(df[df['Cluster'] == cid])
    print(f'  Cluster {cid}: {name} ({count} customers)')

In [ ]:
# Cluster distribution bar chart
cluster_counts = df['Cluster'].value_counts().sort_index()

fig = go.Figure(go.Bar(
    x=[f'Cluster {i}' for i in cluster_counts.index],
    y=cluster_counts.values,
    marker_color=cluster_colors,
    text=cluster_counts.values,
    textposition='outside'
))

fig.update_layout(
    title='📊 Customers per Cluster',
    template='plotly_dark',
    paper_bgcolor='rgba(0,0,0,0)',
    xaxis_title='Cluster',
    yaxis_title='Number of Customers',
    height=400
)
fig.show()

## Step 8: Save Model Artifacts

In [ ]:
import os

# Create models directory if it doesn't exist
os.makedirs('../models', exist_ok=True)

# Save the trained model
joblib.dump(kmeans, '../models/kmeans_model.pkl')
print('✅ Model saved to: models/kmeans_model.pkl')

# Save the scaler
joblib.dump(scaler, '../models/scaler.pkl')
print('✅ Scaler saved to: models/scaler.pkl')

# Verify by loading
loaded_model = joblib.load('../models/kmeans_model.pkl')
loaded_scaler = joblib.load('../models/scaler.pkl')
print(f'\n✅ Verification: Model loaded with {loaded_model.n_clusters} clusters')

In [ ]:
# Test prediction with a sample customer
sample_income = 60
sample_spending = 50

sample = np.array([[sample_income, sample_spending]])
sample_scaled = loaded_scaler.transform(sample)
predicted_cluster = loaded_model.predict(sample_scaled)[0]

print(f'🧪 Test Prediction:')
print(f'   Income: ${sample_income}K, Spending Score: {sample_spending}')
print(f'   Predicted Cluster: {predicted_cluster}')
print(f'   Segment: {cluster_names.get(predicted_cluster, "Unknown")}')

## Summary

### Key Findings:
1. **Optimal K = 5** — confirmed by both Elbow Method and Silhouette Analysis
2. **5 distinct customer segments** identified with clear Income-Spending patterns
3. **Low correlation** between Income and Spending Score — ideal for segmentation
4. **Gender distribution** is relatively balanced across all clusters

### Business Impact:
- **High Income, High Spenders** → VIP programs, luxury products
- **High Income, Low Spenders** → Personalized marketing to increase engagement
- **Low Income, High Spenders** → Retention through rewards and loyalty
- **Low Income, Low Spenders** → Budget-friendly products and essentials
- **Moderate Spenders** → Seasonal promotions and value bundles

### Next Steps:
- Deploy the model via the **Streamlit web app** (`streamlit run app.py`)
- Experiment with additional features (Age, Gender) for multi-dimensional clustering
- Try other algorithms: DBSCAN, Hierarchical Clustering